# Citation Prediction Data Preparation
## Merging Scopus and SciVal Data

This notebook merges publication data from two sources:
- **Scopus file**: Contains abstracts and other metadata
- **SciVal file**: Contains citation metrics and institutional data

Both files will be matched using the **EID** (Electronic Identifier) column.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries loaded successfully!")

## 1. File Setup

**Instructions**: 
1. Place your data files in a `data/` subdirectory
2. Update the filenames below to match your actual files
3. Supported formats: CSV, Excel (.xlsx), TSV

In [ ]:
# File paths - UPDATE THESE TO MATCH YOUR FILES
scopus_file = "data/scopus_data.csv"  # File containing abstracts
scival_file = "data/scival_data.csv"  # File containing citation metrics

# Check if files exist
if os.path.exists(scopus_file):
    print(f"✓ Found Scopus file: {scopus_file} ({os.path.getsize(scopus_file) / (1024**2):.2f} MB)")
else:
    print(f"✗ Scopus file not found: {scopus_file}")

if os.path.exists(scival_file):
    print(f"✓ Found SciVal file: {scival_file} ({os.path.getsize(scival_file) / (1024**2):.2f} MB)")
else:
    print(f"✗ SciVal file not found: {scival_file}")

## 2. Load Data Files

This section loads the data files efficiently, handling different formats automatically.

In [ ]:
def load_data_file(filepath, description=""):
    """
    Load data file automatically detecting format (CSV, TSV, Excel)
    """
    print(f"\nLoading {description}...")
    
    file_ext = Path(filepath).suffix.lower()
    
    try:
        if file_ext == '.csv':
            # Try comma first, then tab if that fails
            try:
                df = pd.read_csv(filepath, low_memory=False)
            except:
                df = pd.read_csv(filepath, sep='\t', low_memory=False)
        elif file_ext in ['.xlsx', '.xls']:
            df = pd.read_excel(filepath)
        elif file_ext == '.tsv':
            df = pd.read_csv(filepath, sep='\t', low_memory=False)
        else:
            raise ValueError(f"Unsupported file format: {file_ext}")
        
        print(f"  ✓ Loaded {len(df):,} rows and {len(df.columns)} columns")
        print(f"  Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
        
        return df
    
    except Exception as e:
        print(f"  ✗ Error loading file: {e}")
        return None

# Load both files
scopus_df = load_data_file(scopus_file, "Scopus data")
scival_df = load_data_file(scival_file, "SciVal data")

## 3. Data Exploration

Let's examine the structure of both datasets and identify the EID columns.

In [ ]:
# Display Scopus data info
if scopus_df is not None:
    print("=" * 80)
    print("SCOPUS DATA SUMMARY")
    print("=" * 80)
    print(f"\nShape: {scopus_df.shape}")
    print(f"\nColumns ({len(scopus_df.columns)}):")
    print(scopus_df.columns.tolist())
    print(f"\nFirst few rows:")
    display(scopus_df.head(3))
    
    # Look for EID column
    eid_cols = [col for col in scopus_df.columns if 'eid' in col.lower()]
    print(f"\nPotential EID columns: {eid_cols}")
    
    # Look for abstract column
    abstract_cols = [col for col in scopus_df.columns if 'abstract' in col.lower()]
    print(f"Abstract columns: {abstract_cols}")

In [ ]:
# Display SciVal data info
if scival_df is not None:
    print("=" * 80)
    print("SCIVAL DATA SUMMARY")
    print("=" * 80)
    print(f"\nShape: {scival_df.shape}")
    print(f"\nColumns ({len(scival_df.columns)}):")
    print(scival_df.columns.tolist())
    print(f"\nFirst few rows:")
    display(scival_df.head(3))
    
    # Look for EID column
    eid_cols = [col for col in scival_df.columns if 'eid' in col.lower()]
    print(f"\nPotential EID columns: {eid_cols}")
    
    # Look for citation columns
    citation_cols = [col for col in scival_df.columns if 'citation' in col.lower() or 'cited' in col.lower()]
    print(f"Citation columns: {citation_cols}")

## 4. Identify and Standardize EID Columns

Before merging, we need to ensure both datasets have a common EID column with matching formats.

In [ ]:
# UPDATE THESE COLUMN NAMES BASED ON YOUR DATA
# After running the cells above, set these to the actual column names
scopus_eid_col = 'EID'  # Column name for EID in Scopus file
scival_eid_col = 'EID'  # Column name for EID in SciVal file
abstract_col = 'Abstract'  # Column name for abstract in Scopus file

# Check if columns exist
if scopus_df is not None and scival_df is not None:
    print("Checking column names...")
    
    if scopus_eid_col not in scopus_df.columns:
        print(f"⚠ Warning: '{scopus_eid_col}' not found in Scopus data")
        print(f"Available columns: {scopus_df.columns.tolist()}")
    else:
        print(f"✓ Found '{scopus_eid_col}' in Scopus data")
    
    if scival_eid_col not in scival_df.columns:
        print(f"⚠ Warning: '{scival_eid_col}' not found in SciVal data")
        print(f"Available columns: {scival_df.columns.tolist()}")
    else:
        print(f"✓ Found '{scival_eid_col}' in SciVal data")
    
    if abstract_col not in scopus_df.columns:
        print(f"⚠ Warning: '{abstract_col}' not found in Scopus data")
    else:
        print(f"✓ Found '{abstract_col}' in Scopus data")

In [ ]:
# Standardize EID columns (remove whitespace, convert to string)
if scopus_df is not None and scival_df is not None:
    print("Standardizing EID columns...")
    
    # Clean and standardize EID in both datasets
    scopus_df['EID_clean'] = scopus_df[scopus_eid_col].astype(str).str.strip()
    scival_df['EID_clean'] = scival_df[scival_eid_col].astype(str).str.strip()
    
    # Check for duplicates
    scopus_dupes = scopus_df['EID_clean'].duplicated().sum()
    scival_dupes = scival_df['EID_clean'].duplicated().sum()
    
    print(f"\nScopus EID stats:")
    print(f"  Total EIDs: {len(scopus_df):,}")
    print(f"  Unique EIDs: {scopus_df['EID_clean'].nunique():,}")
    print(f"  Duplicates: {scopus_dupes:,}")
    
    print(f"\nSciVal EID stats:")
    print(f"  Total EIDs: {len(scival_df):,}")
    print(f"  Unique EIDs: {scival_df['EID_clean'].nunique():,}")
    print(f"  Duplicates: {scival_dupes:,}")
    
    # Check overlap
    overlap = set(scopus_df['EID_clean']) & set(scival_df['EID_clean'])
    print(f"\nOverlapping EIDs: {len(overlap):,}")
    print(f"Overlap rate: {len(overlap) / len(scival_df) * 100:.1f}% of SciVal records")

## 5. Merge Data

Now we'll merge the datasets, adding abstracts from Scopus to the SciVal data.

In [ ]:
if scopus_df is not None and scival_df is not None:
    print("Merging datasets...\n")
    
    # Select only EID and abstract from Scopus
    scopus_subset = scopus_df[['EID_clean', abstract_col]].copy()
    scopus_subset = scopus_subset.rename(columns={abstract_col: 'Abstract'})
    
    # Remove duplicates from Scopus (keep first occurrence)
    scopus_subset = scopus_subset.drop_duplicates(subset='EID_clean', keep='first')
    
    print(f"Scopus records with abstracts: {len(scopus_subset):,}")
    print(f"SciVal records before merge: {len(scival_df):,}")
    
    # Perform left merge (keep all SciVal records, add abstracts where available)
    merged_df = scival_df.merge(
        scopus_subset,
        left_on='EID_clean',
        right_on='EID_clean',
        how='left',
        suffixes=('', '_scopus')
    )
    
    print(f"\nMerged dataset: {len(merged_df):,} rows")
    print(f"Records with abstracts: {merged_df['Abstract'].notna().sum():,}")
    print(f"Records without abstracts: {merged_df['Abstract'].isna().sum():,}")
    print(f"\nAbstract coverage: {merged_df['Abstract'].notna().sum() / len(merged_df) * 100:.1f}%")
    
    # Clean up temporary column
    merged_df = merged_df.drop(columns=['EID_clean'], errors='ignore')
    
    print("\n✓ Merge complete!")

## 6. Data Quality Checks

In [ ]:
if 'merged_df' in locals():
    print("=" * 80)
    print("MERGED DATA QUALITY REPORT")
    print("=" * 80)
    
    print(f"\nDataset Shape: {merged_df.shape}")
    print(f"Total Records: {len(merged_df):,}")
    print(f"Total Features: {len(merged_df.columns)}")
    
    # Missing values
    print("\nMissing Values Summary:")
    missing = merged_df.isnull().sum()
    missing_pct = (missing / len(merged_df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing_Count': missing[missing > 0],
        'Missing_Pct': missing_pct[missing > 0]
    }).sort_values('Missing_Count', ascending=False)
    
    if len(missing_df) > 0:
        display(missing_df.head(20))
    else:
        print("  ✓ No missing values found!")
    
    # Abstract statistics
    if 'Abstract' in merged_df.columns:
        print("\nAbstract Statistics:")
        valid_abstracts = merged_df['Abstract'].dropna()
        if len(valid_abstracts) > 0:
            abstract_lengths = valid_abstracts.str.len()
            print(f"  Mean length: {abstract_lengths.mean():.0f} characters")
            print(f"  Median length: {abstract_lengths.median():.0f} characters")
            print(f"  Min length: {abstract_lengths.min():.0f} characters")
            print(f"  Max length: {abstract_lengths.max():.0f} characters")
    
    # Preview merged data
    print("\nSample of Merged Data:")
    display(merged_df.head(5))

## 7. Save Merged Dataset

In [ ]:
# Save merged dataset
if 'merged_df' in locals():
    output_file = "data/merged_citation_data.csv"
    
    # Create data directory if it doesn't exist
    os.makedirs("data", exist_ok=True)
    
    print(f"Saving merged dataset to {output_file}...")
    merged_df.to_csv(output_file, index=False)
    
    file_size = os.path.getsize(output_file) / (1024**2)
    print(f"\n✓ Saved successfully!")
    print(f"  File: {output_file}")
    print(f"  Size: {file_size:.2f} MB")
    print(f"  Records: {len(merged_df):,}")
    print(f"  Columns: {len(merged_df.columns)}")
    
    # Also save a backup in Excel format (if dataset is not too large)
    if len(merged_df) < 1000000:  # Excel has a limit
        excel_file = "data/merged_citation_data.xlsx"
        print(f"\nSaving Excel backup to {excel_file}...")
        merged_df.to_excel(excel_file, index=False, engine='openpyxl')
        print("✓ Excel backup saved!")

## 8. Next Steps

Now that your data is merged, you can proceed with:

1. **Data Cleaning**: Handle missing values, remove duplicates, fix encoding issues
2. **Feature Engineering**: 
   - Extract author metrics (h-index, citation counts)
   - Venue prestige scores
   - Text features from abstracts (TF-IDF, embeddings)
3. **Exploratory Data Analysis**: 
   - Citation distribution analysis
   - Temporal trends
   - Field-specific patterns
4. **Model Development**: 
   - Classification (high vs low impact)
   - Regression (citation count prediction)

The merged dataset is ready for the next phase of your capstone project!

In [ ]:
# Final summary
if 'merged_df' in locals():
    print("=" * 80)
    print("MERGE PROCESS COMPLETE")
    print("=" * 80)
    print(f"\n✓ Successfully merged Scopus and SciVal data")
    print(f"✓ Final dataset: {len(merged_df):,} publications")
    print(f"✓ Abstract coverage: {merged_df['Abstract'].notna().sum() / len(merged_df) * 100:.1f}%")
    print(f"\nReady for next phase: Feature Engineering and Model Development")